# Model 1: Hybrid GRU + Dense network
### Will the product be out of stock **all day tomorrow**? (binary classification)

**Data:** FreshRetailNet-50K, frozen in `Dataset.ipynb` (Step 7) → `data/train_set.csv`, `data/test_set.csv` (Model 2 uses the exact same rows).
**Target (`label`):** 0 = available at least part of the day · 1 = out of stock all 16 opening hours on day t+1.

**How to read this notebook**
- 📘 **What** explains the step, then the code follows. Every line is commented.
- 🔁 **Tcr ref** points to the teacher's cell that uses the same pattern:
  - `lec` = `lec/lec.ipynb` (MNIST: build/compile/fit comments, dropout, he_normal + l2, classification report, confusion matrix)
  - `lab2` = `lab1/lab2.ipynb` (cancer: split, StandardScaler, sigmoid output, learning curves, good fit / overfit)
  - `lab3c` = `lab1/lab3.ipynb` (cancer: EarlyStopping, Keras Tuner RandomSearch)
  - `imdb` = `lec/IMDB/imdb.ipynb` (EarlyStopping with best weights, comparing trials, best epoch + gap)
- 📝 **Report** says what to write in that part of the report.
- Run top to bottom, use a **light theme**, and screenshot each code cell **and** its output.

---
## Step 0 – Setup
📘 **What:** install + import the libraries and fix the random seed so every run gives the same numbers.

🔁 Tcr ref: `lab2` [1]–[2], `lab3c` [5]

In [ ]:
# Step 0.1: install the libraries (run once, then you can comment it out)
%pip install pandas numpy matplotlib scikit-learn keras tensorflow keras-tuner

In [ ]:
# Step 0.2: imports + seed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import keras

# set the random seed so the results are reproducible (same numbers every time you run)
keras.utils.set_random_seed(42)

# light theme for the plots (screenshots for the report)
plt.style.use("default")

class_names = ["0: available", "1: out all day"]

---
## Section 1 – Data preparation

### Step 1.1 – Load the frozen data
📘 **What:** load the train/test CSVs saved by `Dataset.ipynb`. The split is **by time** (train period vs the last evaluation week), so the test set is truly "the future".

🔁 Tcr ref: `lab2` [3]

📝 **Report:** number of rows/columns in train and test, and that both models use the same frozen files.

In [ ]:
# Step 1.1: load the train/test sets saved by Dataset.ipynb (same rows for both models)
train_set = pd.read_csv("data/train_set.csv")
test_set = pd.read_csv("data/test_set.csv")

print("train shape:", train_set.shape)   # (rows, columns)
print("test shape: ", test_set.shape)
print("missing values:", train_set.isna().sum().sum(), "(train)", test_set.isna().sum().sum(), "(test)")
train_set.head()

### Step 1.2 – Class balance
📘 **What:** check that the target is balanced. `Dataset.ipynb` kept all full-day stockouts and an equal-size sample of the other rows → 50/50, so **accuracy is a fair metric** and random guessing = 50%.

🔁 Tcr ref: `lab2` [5]

In [ ]:
# Step 1.2: class balance (0 = available, 1 = out of stock all day)
print("train:\n", train_set["label"].value_counts().sort_index())
print("test:\n", test_set["label"].value_counts().sort_index())

train_set["label"].value_counts().sort_index().plot(kind="bar", color=["tab:blue", "tab:red"])
plt.title("Class balance (train set)")
plt.xticks([0, 1], class_names, rotation=0)
plt.ylabel("Number of rows")
plt.show()

### Step 1.3 – Split the columns into the 2 inputs of the model
📘 **What:** the model has **two inputs**, one for each type of data:
- **Sequence input** – the last 14 days of 4 values (`sale_amount, stock_hour6_22_cnt, discount, activity_flag`), oldest day first → shape **(14 days, 4 features)**. This goes to the GRU.
- **Static input** – everything else about *today* (hourly sales/stock, weather, holiday, day of week). This goes to a Dense layer.

In [ ]:
# Step 1.3: split the columns into the 2 inputs
SEQ = ["sale_amount", "stock_hour6_22_cnt", "discount", "activity_flag"]

# lag13, lag12, ..., lag1, today  -> oldest day first (the GRU reads them in time order)
seq_cols = [f"{c}_lag{lag}" if lag > 0 else c for lag in range(13, -1, -1) for c in SEQ]

# static = all the other columns except the target
static_cols = [c for c in train_set.columns if c not in seq_cols + ["label"]]

print("sequence columns:", len(seq_cols), "-> 14 days x 4 features")
print("static columns:  ", len(static_cols))

### Step 1.4 – Train / validation / test
📘 **What:** take 20% of the **train** data as a **validation set**. Validation is used to compare trials and tune hyperparameters; the **test set is only used for the final check** (train/test gap must be within 5%).

> The teacher's labs use `validation_data=(X_test, y_test)`. Here we keep a separate validation set so the test set is never used to choose the model → the final test accuracy is honest.

🔁 Tcr ref: `lab2` [6]

In [ ]:
# Step 1.4: hold out 20% of the TRAIN data as a validation set
train_part, val_part = train_test_split(
    train_set,
    test_size=0.2,                   # 20% validation, 80% train
    random_state=42,                 # same shuffle every run
    stratify=train_set["label"]      # keep 50/50 in train and validation
)
y_train = train_part["label"].values
y_val = val_part["label"].values
y_test = test_set["label"].values

print("train:", train_part.shape)
print("val:  ", val_part.shape)
print("test: ", test_set.shape)

### Step 1.5 – Standardization + reshape
📘 **What:** standardize every column (mean 0, std 1) so large columns (e.g. sales) do not dominate small ones (e.g. flags). Then reshape the sequence columns into a 3-D array `(rows, 14, 4)` because a GRU needs **(samples, time steps, features)**.

🔁 Tcr ref: `lab2` [7] – `fit_transform` on train only, `transform` on val/test → no data leakage.

In [ ]:
# Step 1.5: standardization (mean 0, std 1) + reshape the sequence into (rows, 14, 4)
scaler_seq = StandardScaler().fit(train_part[seq_cols])         # learn mean/std from TRAIN only
scaler_static = StandardScaler().fit(train_part[static_cols])

def make_inputs(d):
    seq = scaler_seq.transform(d[seq_cols]).reshape(-1, 14, len(SEQ)).astype("float32")
    static = scaler_static.transform(d[static_cols]).astype("float32")
    return [seq, static]             # the model takes a list of 2 inputs

X_train = make_inputs(train_part)
X_val = make_inputs(val_part)
X_test = make_inputs(test_set)       # use the SAME mean/std -> no data leakage

print("sequence input:", X_train[0].shape, " (rows, 14 days, 4 features)")
print("static input:  ", X_train[1].shape, " (rows, features)")

---
## Section 2 – Initial model

### Step 2.1 – Build
📘 **What:** a **hybrid (multi-input) network**:
- **GRU branch (64 units):** reads the last 14 days *in order*, so it can learn trends such as "stockouts getting longer day after day". GRU is a simpler LSTM (fewer parameters, similar accuracy). A plain Dense layer would treat day 1 and day 14 as unrelated columns.
- **Dense branch (64, ReLU):** today's information that is not a sequence.
- **Concatenate → Dense 64 → Dense 32 (ReLU):** combines both branches.
- **Output Dense(1, sigmoid):** probability of class 1 (binary → sigmoid, like the cancer lab).

We use the **functional API** (`keras.Model(inputs, outputs)`) instead of `Sequential()` because `Sequential` only allows **one** input. Hidden layers: GRU + Dense branch + 2 Dense = **4**. No regularization yet, so we can see if it overfits.

🔁 Tcr ref: `lec` [8], `lab2` [8]

In [ ]:
# Step 2.1: build the initial model
from keras.layers import Input, Dense, GRU, Concatenate, Dropout

seq_in = Input(shape=(14, len(SEQ)), name="last_14_days")       # input 1: 14 days x 4 features
static_in = Input(shape=(len(static_cols),), name="today")      # input 2: today's features

x = GRU(64)(seq_in)                              # branch 1: GRU reads the 14 days in order
y = Dense(64, activation="relu")(static_in)      # branch 2: Dense on today's features
z = Concatenate()([x, y])                        # merge the 2 branches into one vector
z = Dense(64, activation="relu")(z)              # hidden layer 3
z = Dense(32, activation="relu")(z)              # hidden layer 4
out = Dense(1, activation="sigmoid")(z)          # output: probability of "out all day" (0 to 1)

model = keras.Model(inputs=[seq_in, static_in], outputs=out)

In [ ]:
# Step 2.2: summary (layers + number of parameters)
model.summary()

### Step 2.3 – Compile + train
🔁 Tcr ref: `lec` [8], `lab2` [10]–[11]

In [ ]:
# Step 2.3: compile
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# optimizer = 'adam' : meaning we will use the Adam optimizer to update the weights (default lr = 0.001)
# loss = 'binary_crossentropy' : meaning the loss for a yes/no target with a sigmoid output
# metrics = ['accuracy'] : meaning we measure accuracy (fair here because the classes are 50/50)

In [ ]:
# Step 2.4: train
# epochs = 30 : meaning we will go through the entire training set 30 times
# batch_size = 128 : meaning we update the weights after every 128 rows (23,865 / 128 = ~187 updates per epoch)
# validation_data = (X_val, y_val) : meaning we check the model on the validation set after each epoch
history = model.fit(X_train, y_train, epochs=30, batch_size=128,
                    validation_data=(X_val, y_val))

### Step 2.5 – Learning curves + fit diagnosis
📘 **What:** plot train vs validation accuracy/loss, then print the **train/test gap**.
- **Good fit:** both curves close together and flat at the end.
- **Overfit:** train loss keeps going down but validation loss goes **up**; train accuracy clearly higher than validation.
- **Underfit:** both accuracies low and still improving.

🔁 Tcr ref: `lab2` [13]–[15], `lec` [9]–[10], `imdb` [15]

📝 **Report:** screenshot both plots + the gap line, and say good fit / underfit / overfit with the numbers.

In [ ]:
# Step 2.5a: function to plot the learning curves (we reuse it for every model)
def plot_history(history, title):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs = range(1, len(acc) + 1)
    # creating subplots, figsize (width, height) in inches
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 3))
    axes[0].plot(epochs, acc, "b", label="Train Accuracy")
    axes[0].plot(epochs, val_acc, "r", label="Validation Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_title(title + " - accuracy")
    axes[0].legend()

    axes[1].plot(epochs, loss, "b", label="Train Loss")
    axes[1].plot(epochs, val_loss, "r", label="Validation Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].set_title(title + " - loss")
    axes[1].legend()

    # adjusting layout for better spacing
    plt.tight_layout()
    plt.show()

plot_history(history, "Initial model")

In [ ]:
# Step 2.5b: fit diagnosis - train vs test accuracy (brief: the gap must be within 5%)
def gap_check(m, name):
    train_acc = m.evaluate(X_train, y_train, verbose=0)[1]
    val_acc = m.evaluate(X_val, y_val, verbose=0)[1]
    test_acc = m.evaluate(X_test, y_test, verbose=0)[1]
    gap = train_acc - test_acc
    print("%-12s train acc=%.4f  val acc=%.4f  test acc=%.4f  gap=%.4f  -> %s" %
          (name, train_acc, val_acc, test_acc, gap, "OK (within 5%)" if abs(gap) <= 0.05 else "NOT OK (over 5%)"))
    return {"Model": name, "Parameters": m.count_params(), "Train acc": round(train_acc, 4),
            "Val acc": round(val_acc, 4), "Test acc": round(test_acc, 4), "Gap": round(gap, 4)}

gap_check(model, "Initial")

# is it a good fit? underfit? or overfit?
# -> write your answer here after looking at the numbers and the curves, e.g.:
# overfit: train loss keeps going down but validation loss goes UP after a few epochs,
#          and the train accuracy is clearly higher than the test accuracy (gap > 5%).

In [ ]:
# what can we do to improve the model? (same list as the teacher's lec [11])
# 1. add dropout layers to reduce overfitting                          -> Step 3.1
# 2. he_normal initializer + L2 regularization + smaller learning rate -> Step 3.2
# 3. early stopping: stop when validation loss stops decreasing        -> Step 3.2
# 4. hyperparameter tuning: units, layers, dropout, L2, learning rate  -> Section 4
# 5. batch size + learning rate scheduling (ReduceLROnPlateau)         -> Step 4.6 - 4.7

---
## Section 3 – Improve step by step (manual trials)
📘 **What:** change a few things at a time and compare, like the teacher's MNIST/IMDB lessons. Each new line is marked `# NEW`.

🔁 Tcr ref: `lec` [12]–[17], `lab2` [18]–[20], `imdb` [10]–[16]

📝 **Report:** a table of the trials (Step 3.3) and one sentence per trial on what changed and why.

In [ ]:
# Step 3.1: trial B - add Dropout to reduce overfitting
seq_in = Input(shape=(14, len(SEQ)))
static_in = Input(shape=(len(static_cols),))

x = GRU(64, dropout=0.2)(seq_in)                 # NEW: drop 20% of the GRU inputs at random
y = Dense(64, activation="relu")(static_in)
y = Dropout(0.3)(y)                              # NEW: turn off 30% of neurons at random during training
z = Concatenate()([x, y])
z = Dense(64, activation="relu")(z)
z = Dropout(0.3)(z)                              # NEW
z = Dense(32, activation="relu")(z)
z = Dropout(0.3)(z)                              # NEW
out = Dense(1, activation="sigmoid")(z)

model_b = keras.Model(inputs=[seq_in, static_in], outputs=out)
model_b.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history_b = model_b.fit(X_train, y_train, epochs=30, batch_size=128,
                        validation_data=(X_val, y_val))
plot_history(history_b, "Trial B (+Dropout)")

In [ ]:
# Step 3.2: trial C - he_normal + L2 regularization + smaller learning rate + EarlyStopping
from keras import regularizers
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

seq_in = Input(shape=(14, len(SEQ)))
static_in = Input(shape=(len(static_cols),))

x = GRU(64, dropout=0.2,
        kernel_regularizer=regularizers.l2(0.01))(seq_in)          # NEW: L2 punishes large weights
y = Dense(64, activation="relu",
          kernel_initializer="he_normal",                           # NEW: good starting weights for ReLU
          kernel_regularizer=regularizers.l2(0.01))(static_in)      # NEW
y = Dropout(0.3)(y)
z = Concatenate()([x, y])
z = Dense(64, activation="relu", kernel_initializer="he_normal",
          kernel_regularizer=regularizers.l2(0.01))(z)              # NEW
z = Dropout(0.3)(z)
z = Dense(32, activation="relu", kernel_initializer="he_normal")(z)
z = Dropout(0.3)(z)
out = Dense(1, activation="sigmoid")(z)

model_c = keras.Model(inputs=[seq_in, static_in], outputs=out)
model_c.compile(optimizer=Adam(learning_rate=0.0005),               # NEW: lr 0.001 -> 0.0005 (smaller, steadier steps)
                loss="binary_crossentropy", metrics=["accuracy"])

# NEW: stop when val_loss has not improved for 5 epochs, and go back to the best weights
early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_c = model_c.fit(X_train, y_train, epochs=100, batch_size=128,
                        validation_data=(X_val, y_val), callbacks=[early])
plot_history(history_c, "Trial C (+he_normal +L2 +lr +EarlyStopping)")

In [ ]:
# Step 3.3a: compare the validation curves of all trials (teacher's imdb [17])
runs = {"A initial": history,
        "B +Dropout": history_b,
        "C +L2 +he_normal +lr +EarlyStop": history_c}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, h in runs.items():
    ax1.plot(h.history["val_accuracy"], label=name)
    ax2.plot(h.history["val_loss"], label=name)

ax1.set_title("Validation accuracy (higher = better)")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Val accuracy"); ax1.legend()
ax2.set_title("Validation loss (lower = better)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Val loss"); ax2.legend()
plt.show()

In [ ]:
# Step 3.3b: trial summary table (train / val / test accuracy + gap)
trial_table = pd.DataFrame([gap_check(model, "A initial"),
                            gap_check(model_b, "B +Dropout"),
                            gap_check(model_c, "C +L2 +lr +ES")])
trial_table

---
## Section 4 – Hyperparameter tuning (Keras Tuner)
📘 **What:** instead of guessing, let `RandomSearch` try 10 random combinations around the trial C design and keep the one with the best **validation** accuracy.

| Hyperparameter | Search space | Why |
|---|---|---|
| GRU units | 32 to 128 (step 32) | how much the sequence branch can remember |
| Dense branch units | 32 to 128 (step 32) | size of the "today" branch |
| Number of Dense layers after merging | 2 to 3 | depth of the combined part |
| Units per merged layer | 16 to 128 (step 16) | width of each layer |
| Activation | relu, tanh | same choice the teacher tunes in `lab3c` |
| Dropout | 0.3 to 0.5 (step 0.1) | less overfitting (trial A overfit) |
| L2 regularization | 0.005, 0.01, 0.02 | keeps the train/test gap small |
| Learning rate | 0.01, 0.001, 0.0001 | size of each weight update |
| Batch size | 32, 64, 128, 256 | tuned separately in Step 4.5 |
| Epochs | EarlyStopping | stop when val loss stops improving, keep the best weights |

🔁 Tcr ref: `lab3c` [6]–[8], [10]

📝 **Report:** this table, the `results_summary` screenshot and the best hyperparameters.

In [ ]:
# Step 4.1: model-building function (hp = the hyperparameters the tuner chooses)
import keras_tuner as kt
from keras_tuner import RandomSearch

def build_model(hp):
    l2 = regularizers.l2(hp.Choice("l2", values=[5e-3, 1e-2, 2e-2]))
    dropout = hp.Float("dropout", 0.3, 0.5, step=0.1)
    activation = hp.Choice("activation", values=["relu", "tanh"])

    seq_in = Input(shape=(14, len(SEQ)))
    static_in = Input(shape=(len(static_cols),))

    # branch 1: GRU over the last 14 days (tune the number of units)
    x = GRU(hp.Int("gru_units", 32, 128, step=32), dropout=0.2, kernel_regularizer=l2)(seq_in)

    # branch 2: Dense on today's features (tune the number of units)
    y = Dense(hp.Int("dense_units", 32, 128, step=32), activation=activation,
              kernel_initializer="he_normal", kernel_regularizer=l2)(static_in)
    y = Dropout(dropout)(y)

    # merge + tune the NUMBER of Dense layers (2 to 3) and the units of each one
    z = Concatenate()([x, y])
    for i in range(hp.Int("num_layers", 2, 3)):
        z = Dense(hp.Int(f"units_{i}", 16, 128, step=16), activation=activation,
                  kernel_initializer="he_normal", kernel_regularizer=l2)(z)
        z = Dropout(dropout)(z)
    out = Dense(1, activation="sigmoid")(z)

    model = keras.Model(inputs=[seq_in, static_in], outputs=out)
    model.compile(optimizer=Adam(learning_rate=hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4])),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
# Step 4.2: create the tuner
tuner = RandomSearch(
    build_model,
    objective="val_accuracy",     # pick the model with the best validation accuracy
    max_trials=10,                # try 10 random combinations
    executions_per_trial=1,       # teacher uses 3; 1 here because each GRU trial is slow (set 3 if you have time)
    directory="my_dir",           # trials are saved here (ignored by git)
    project_name="gru_stockout",
    overwrite=True                # start a fresh search every run
)
tuner.search_space_summary()

In [ ]:
# Step 4.3: search (validation set only, the test set is NOT used here)
tuner.search(X_train, y_train, epochs=30, batch_size=128,
             validation_data=(X_val, y_val),
             callbacks=[EarlyStopping(monitor="val_loss", patience=5)])

In [ ]:
# Step 4.4: showing the 10 best trials + the best hyperparameters
tuner.results_summary(num_trials=10)

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\nBest hyperparameters:", best_hp.values)

# same results as one table (easier to screenshot)
rows = []
for t in tuner.oracle.get_best_trials(num_trials=10):
    row = dict(t.hyperparameters.values)
    row["val_accuracy"] = round(t.score, 4)
    rows.append(row)
pd.DataFrame(rows)

In [ ]:
# Step 4.5: tune the batch size (and find the best number of epochs with EarlyStopping)
results = []
for bs in [32, 64, 128, 256]:
    keras.utils.set_random_seed(42)             # same starting weights for every batch size -> fair comparison
    m = tuner.hypermodel.build(best_hp)         # fresh model with the best hyperparameters
    h = m.fit(X_train, y_train, epochs=100, batch_size=bs,
              validation_data=(X_val, y_val), verbose=0,
              callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])
    best = int(np.argmin(h.history["val_loss"]))         # epoch with the lowest val_loss
    results.append({"batch_size": bs,
                    "best_epoch": best + 1,
                    "train_acc": round(h.history["accuracy"][best], 4),
                    "val_acc": round(h.history["val_accuracy"][best], 4)})
    print("batch size", bs, "done")

bs_results = pd.DataFrame(results)
best_bs = int(bs_results.loc[bs_results["val_acc"].idxmax(), "batch_size"])
print("Best batch size:", best_bs)
bs_results

In [ ]:
# Step 4.6: FINAL tuned model = best hyperparameters + best batch size
from keras.callbacks import ReduceLROnPlateau

keras.utils.set_random_seed(42)
model_final = tuner.hypermodel.build(best_hp)
model_final.summary()

# EarlyStopping : stop when val_loss has not improved for 8 epochs and keep the best weights
# ReduceLROnPlateau : learning rate scheduling -> halve the learning rate when val_loss stops improving for 3 epochs
history_final = model_final.fit(X_train, y_train, epochs=100, batch_size=best_bs,
                                validation_data=(X_val, y_val),
                                callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
                                           ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)])

In [ ]:
# Step 4.7: best epoch + gap of the tuned model (teacher's imdb [15])
best = int(np.argmin(history_final.history["val_loss"]))
print("best epoch :", best + 1)
print("train acc  :", round(history_final.history["accuracy"][best], 4))
print("val acc    :", round(history_final.history["val_accuracy"][best], 4))
print()
gap_check(model_final, "Tuned")

---
## Section 5 – Evaluation on the TEST set
🔁 Tcr ref: `lec` [18]–[22]

📝 **Report:** learning curves (initial vs tuned), classification report, confusion matrix, final metrics table vs the 2 baselines.

In [ ]:
# Step 5.1: learning curves (initial vs tuned)
plot_history(history, "Initial model")
plot_history(history_final, "Tuned model")

In [ ]:
# Step 5.2: prediction
prediction_x = model_final.predict(X_test, verbose=0).ravel()   # probability of class 1 for every row
predict1 = (prediction_x > 0.5).astype(int)                     # convert output to class label
# sigmoid gives ONE probability, so we use "> 0.5" instead of argmax (argmax is for softmax with many outputs)
print(predict1[:20])

In [ ]:
# Step 5.3: precision, recall, f1-score and accuracy (initial vs tuned)
from sklearn.metrics import classification_report

predict_initial = (model.predict(X_test, verbose=0).ravel() > 0.5).astype(int)

print("===== Initial model =====")
print(classification_report(y_test, predict_initial, target_names=class_names, digits=4))
print("===== Tuned model =====")
print(classification_report(y_test, predict1, target_names=class_names, digits=4))

In [ ]:
# Step 5.4: confusion matrix of the tuned model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# compute confusion matrix
cm = confusion_matrix(y_test, predict1)

# display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (tuned model, test set)")
plt.show()

# the diagonal values are the correct predictions, the off-diagonal values are the incorrect predictions
# top-right = false alarm (predicted out all day, but it was available)
# bottom-left = missed stockout (predicted available, but it was out all day)

In [ ]:
# Step 5.5: final metrics table (initial vs tuned vs 2 baselines)
from sklearn.metrics import f1_score, roc_auc_score

def final_row(name, m):
    train_acc = m.evaluate(X_train, y_train, verbose=0)[1]
    test_acc = m.evaluate(X_test, y_test, verbose=0)[1]
    prob = m.predict(X_test, verbose=0).ravel()
    return {"Model": name, "Parameters": m.count_params(),
            "Train acc": round(train_acc, 4), "Test acc": round(test_acc, 4),
            "Gap": round(train_acc - test_acc, 4),
            "F1": round(f1_score(y_test, (prob > 0.5).astype(int)), 4),
            "ROC AUC": round(roc_auc_score(y_test, prob), 4)}

# baseline 1: random guessing -> 50% because the classes are balanced
random_guess = {"Model": "Baseline: random guess", "Parameters": 0,
                "Train acc": 0.5, "Test acc": 0.5, "Gap": 0.0, "F1": 0.5, "ROC AUC": 0.5}

# baseline 2: "tomorrow = today" -> predict out all day if the product was out all 16 hours today
def same_as_today(d):
    return (d["stock_hour6_22_cnt"] >= 16).astype(int)

same_today = {"Model": "Baseline: same as today", "Parameters": 0,
              "Train acc": round((same_as_today(train_part) == y_train).mean(), 4),
              "Test acc": round((same_as_today(test_set) == y_test).mean(), 4),
              "Gap": None, "F1": round(f1_score(y_test, same_as_today(test_set)), 4), "ROC AUC": None}

metrics_table = pd.DataFrame([final_row("Initial hybrid GRU", model),
                              final_row("Tuned hybrid GRU", model_final),
                              random_guess, same_today])
metrics_table

### Step 5.6 – Critical discussion
📝 *(write in your own words, using the numbers above)*
- **Accuracy vs baselines:** how much better than random guessing (50%) and "same as today"?
- **Gap:** is train − test within 5%? What did dropout, L2 and early stopping change compared with the initial model (Step 3.3 table)?
- **Confusion matrix:** which mistake is more common – missed full-day stockouts (false negatives) or false alarms (false positives)? Which one costs a store more?
- **Why the hybrid design:** what does the GRU branch (last 14 days in order) add that a Dense layer on today's data cannot?
- **Limitations:** restocking decisions are not in the data; balancing to 50/50 changes the real class mix (~4% full-day stockouts); only one week of test data.
- **Improvements:** more series, longer history window, product/store embeddings, comparing with Model 2.

---
## Section 6 – Save the final model

In [ ]:
# Step 6.1: save the tuned model + the final metrics table
model_final.save("model1_hybrid_gru_tuned.keras")
metrics_table.to_csv("model1_results.csv", index=False)

print("Final tuned hybrid GRU")
print("  test accuracy: ", metrics_table.loc[1, "Test acc"])
print("  train/test gap:", metrics_table.loc[1, "Gap"])
print("  best hyperparameters:", best_hp.values, " batch size:", best_bs)